# 02 — Table Q&A (main system)

Goal: natural-language questions over the cleaned DataFrame using **LlamaIndex + pandas + Gemini**.

The model should generate dataframe operations; pandas should compute the aggregation.
Load `GOOGLE_API_KEY` from `.env`. Do not hardcode keys.

This notebook is the product path, not RAG.


In [1]:
import os
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.evaluation import load_gold_set
from src.gemini import build_pandas_query_engine, load_project_env

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 80)

load_project_env(ROOT / ".env")
print(f"Repo root: {ROOT}")
print(f"GOOGLE_API_KEY present: {bool(os.getenv('GOOGLE_API_KEY') or os.getenv('GEMINI_API_KEY'))}")


Repo root: /Users/maato4/Library/CloudStorage/OneDrive-LeviStrauss&Co/Documents/GitHub/agentic_ai_learning/retail-analytics-qa
GOOGLE_API_KEY present: True


## Load the processed table

The cleaned file is stored as a pickled DataFrame in `data/processed/`. It has the revenue and cancellation flags we used during the cleaning notebook.

In [2]:
processed_path = ROOT / "data" / "processed" / "online_retail_2009_2010.pkl"
if not processed_path.exists():
    raise FileNotFoundError(f"Processed data not found: {processed_path}")

df = pd.read_pickle(processed_path)
df = df.copy()
df['Customer ID'] = df['Customer ID'].astype('Int64')

print('shape:', df.shape)
print('date range:', df['InvoiceDate'].min(), '→', df['InvoiceDate'].max())
print('non-guest customers:', df.loc[~df['is_guest'], 'Customer ID'].nunique())
display(df.head())


shape: (525461, 14)
date range: 2009-12-01 07:45:00 → 2010-12-09 20:01:00
non-guest customers: 4383


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,is_cancellation,is_guest,is_bad_debt,is_test,is_non_product,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,False,False,False,False,False,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,False,False,False,False,False,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,False,False,False,False,False,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,False,False,False,False,False,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,False,False,False,False,False,30.0


## Define a small benchmark set

These are realistic business questions over the table. They are designed to exercise aggregations, not text lookup.

We compute the numeric target directly in pandas so the notebook has a clean ground truth to compare against.

In [3]:
benchmark_questions = load_gold_set(ROOT / "eval" / "gold_generation.jsonl")

for item in benchmark_questions:
    print(item["question"])
    print("  ground truth:", item["expected_display"])


Which identified customer generated the most gross sales revenue before returns?
  ground truth: Customer 18102, revenue 349,164.35
What was the average net customer lifetime value for identified customers?
  ground truth: 1,950.56
Which country generated the most net revenue across all transaction lines?
  ground truth: United Kingdom, revenue 8,194,777.53
What was the largest single positive line revenue?
  ground truth: 25,111.09
How many identified customers made at least one positive sale?
  ground truth: 4,312
What was total net revenue across all transaction lines, including returns?
  ground truth: 9,539,484.63


## Run the queries through Gemini + PandasQueryEngine

The LLM is instructed to generate a Pandas expression, and the DataFrame does the actual math. This is the core architecture for aggregations over a table.

In [ ]:
query_engine = build_pandas_query_engine(
    df,
    model=os.getenv('GEMINI_CHAT_MODEL', 'gemini-3.5-flash-lite'),
    temperature=0.0,
    max_tokens=256,
    verbose=False,
    head=20,
)

In [ ]:

for index, item in enumerate(benchmark_questions, start=1):
    answer = await query_engine.aquery(item['question'])
    print(f"\n{index}. {item['question']}")
    print('answer:', answer.response)
    print('ground truth:', item['expected_display'])


INFO:google_genai.chats:AFC is enabled with max remote calls: 10.
AFC is enabled with max remote calls: 10.
> Pandas Instructions:
```
df[~df['is_cancellation']].groupby('Customer ID')['Revenue'].sum().idxmax()
```
> Pandas Output: 18102

1. Which identified customer generated the most gross sales revenue before returns?
answer: 18102
ground truth: Customer 18102, revenue 349,164.35
INFO:google_genai.chats:AFC is enabled with max remote calls: 10.
AFC is enabled with max remote calls: 10.
> Pandas Instructions:
```
df[df['Customer ID'].notna()].groupby('Customer ID')['Revenue'].sum().mean()
```
> Pandas Output: 1904.679117955738

2. What was the average net customer lifetime value for identified customers?
answer: 1904.679117955738
ground truth: 1,950.56
INFO:google_genai.chats:AFC is enabled with max remote calls: 10.
AFC is enabled with max remote calls: 10.
> Pandas Instructions:
```
df.groupby('Country')['Revenue'].sum().idxmax()
```
> Pandas Output: United Kingdom

3. Which countr

## Why this pattern matters

Table Q&A works best when the model translates a business question into a trusted dataframe operation instead of estimating from a handful of retrieved rows.

This keeps the answer grounded in the actual data and makes evaluation straightforward.